# 13. 생성 모델

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_13_%EC%83%9D%EC%84%B1_%EB%AA%A8%EB%8D%B8.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실습: VAE와 조건부 GAN

### 🔬 실습 — VAE 구현과 학습

① **VAE 모델.** 인코더가 평균 μ와 로그분산 log σ²을 출력하고, 재파라미터화로 잠재변수 z를 샘플링한다.


In [ ]:
import torch, torch.nn as nn

class VAE(nn.Module):
    def __init__(self, in_dim=784, h=400, z_dim=20):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h)
        self.fc_mu     = nn.Linear(h, z_dim)
        self.fc_logvar = nn.Linear(h, z_dim)
        self.dec = nn.Sequential(nn.Linear(z_dim, h), nn.ReLU(),
                                 nn.Linear(h, in_dim), nn.Sigmoid())
    def encode(self, x):
        h = torch.relu(self.fc1(x)); return self.fc_mu(h), self.fc_logvar(h)
    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar); eps = torch.randn_like(std)
        return mu + std * eps                       # 재파라미터화 트릭
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        return self.dec(z), mu, logvar

② **ELBO 손실.** 재구성(BCE) + KL 닫힌 형태.


In [ ]:
import torch.nn.functional as F
def vae_loss(recon, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(recon, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl

③ **학습 루프.** 표준 학습 루프로 ELBO 최소화.


In [ ]:
from torchvision import datasets, transforms
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST("data", train=True, download=True,
                   transform=transforms.ToTensor()),   # [0,1] 정규화
    batch_size=128, shuffle=True)

torch.manual_seed(0)                       # 재현성(1장)
model = VAE(); optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(10):
    total = 0.0
    for x, _ in train_loader:
        x = x.view(x.size(0), -1)        # (B,1,28,28) -> (B,784)
        recon, mu, logvar = model(x)
        loss = vae_loss(recon, x, mu, logvar)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    print(f"ep{epoch+1:02d} -ELBO(샘플당) "
          f"{total/len(train_loader.dataset):.1f}")

**실행 결과 예시**

```
ep01 -ELBO(샘플당) 163.3
ep05 -ELBO(샘플당) 109.6
ep10 -ELBO(샘플당) 106.1
생성 샘플: torch.Size([16, 784])
```

(매 에포크 한 줄씩 출력되며, 위는 그중 세 줄이다 — 본문 코드를 시드 0으로 그대로 실행한 결과.)


④ **새 샘플 생성.** 사전 분포에서 \(z\)를 뽑아 디코더로 복원.


In [ ]:
model.eval()
with torch.no_grad():
    z = torch.randn(16, 20)              # N(0,I)에서 샘플링
    samples = model.dec(z)               # (16, 784) 새 이미지
print("생성 샘플:", samples.shape)

### 🔬 실습 — 조건부 GAN: "원하는 클래스"를 지정해 생성하기

① **모델.** 레이블을 임베딩해 입력에 이어 붙인다(concatenate).


In [ ]:
import torch, torch.nn as nn

class CGenerator(nn.Module):
    def __init__(self, z_dim=64, n_cls=10, emb=16):
        super().__init__()
        self.label_emb = nn.Embedding(n_cls, emb)     # 레이블 -> 벡터
        self.net = nn.Sequential(
            nn.Linear(z_dim + emb, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 784), nn.Tanh())           # 출력 [-1,1]
    def forward(self, z, y):
        zy = torch.cat([z, self.label_emb(y)], dim=1) # 노이즈+조건 결합
        return self.net(zy)

class CDiscriminator(nn.Module):
    def __init__(self, n_cls=10, emb=16):
        super().__init__()
        self.label_emb = nn.Embedding(n_cls, emb)
        self.net = nn.Sequential(
            nn.Linear(784 + emb, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1))                        # 진짜/가짜 로짓
    def forward(self, x, y):
        xy = torch.cat([x, self.label_emb(y)], dim=1) # 이미지+조건 결합
        return self.net(xy)

② **번갈아 학습.** 13.2절의 minimax를 그대로, 조건만 추가된다.


In [ ]:
from torchvision import datasets, transforms
tfm = transforms.Compose([transforms.ToTensor(),
                          transforms.Normalize(0.5, 0.5)])  # [-1,1] 정규화
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST("data", train=True, download=True, transform=tfm),
    batch_size=128, shuffle=True)

torch.manual_seed(0)                       # 재현성(1장)
G, D = CGenerator(), CDiscriminator()
opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

for epoch in range(10):
    for x, y in train_loader:
        x = x.view(x.size(0), -1)           # (B,1,28,28) -> (B,784)
        B = x.size(0)
        real, fake = torch.ones(B, 1), torch.zeros(B, 1)
        # --- 판별기: 진짜는 진짜로, 가짜는 가짜로 ---
        z = torch.randn(B, 64)
        x_fake = G(z, y).detach()           # G 고정(detach)하고 D만 학습
        d_loss = bce(D(x, y), real) + bce(D(x_fake, y), fake)
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()
        # --- 생성기: 가짜를 진짜로 속이도록 ---
        z = torch.randn(B, 64)
        g_loss = bce(D(G(z, y), y), real)   # D를 속이는 방향
        opt_g.zero_grad(); g_loss.backward(); opt_g.step()
    print(f"ep{epoch+1:02d} d_loss {d_loss.item():.2f} "
          f"| g_loss {g_loss.item():.2f}")

③ **원하는 숫자만 생성.** 조건의 힘을 확인한다.


In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(8, 64)
    y = torch.full((8,), 7)                 # "7"만 8장 생성
    samples = G(z, y)

**실행 결과 예시**

```
ep10 d_loss 0.31 | g_loss 2.68 — 지정한 7 형태의 샘플이 생성된다
```

(본문 코드를 시드 0으로 그대로 실행한 결과이다. 에포크를 10으로 둔 데에는 이유가 있다 — 이 단순한 MLP 구성으로 30 에포크까지 돌리면 판별기가 압승해 d_loss가 0으로 떨어지고 생성기가 더 배우지 못하는 균형 실패가 흔히 나타난다. 직접 늘려서 확인해 보라.)


---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
